# 03. Обучение модели спроса и выручки с погодой

В этом ноутбуке мы обучаем модель, которая по району, дате, часу и погоде предсказывает:

- `demand` — сколько поездок будет в районе;
- `revenue` — какая будет суммарная выручка.

Важная идея: спрос нельзя нормально предсказывать по одной отдельной поездке.  
Поэтому сначала мы превращаем исходную таблицу поездок в таблицу вида:

```text
date_hour + PULocationID -> demand + revenue
```

То есть одна строка = один район в один конкретный час.


In [ ]:
# Если какие-то библиотеки не установлены, сначала запусти в терминале:
# pip install pandas pyarrow scikit-learn joblib matplotlib

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib
import matplotlib.pyplot as plt


## 1. Настраиваем пути

Так как ноутбук лежит в папке `notebooks`, корень проекта — это папка на уровень выше.

Структура у тебя такая:

```text
NYC_TAXI_PROJECT/
├── data/
│   ├── raw/
│   │   ├── open-meteo-40.74N74.04W32m.csv
│   │   └── open-meteo-40.74N74.04W32m (1).csv
│   └── processed/
│       └── my_clean_2.parquet
├── notebooks/
│   └── 03_model_training_with_weather.ipynb
├── reports/
│   ├── figures/
│   └── tables/
└── models/
```


In [ ]:
# Корень проекта.
# Если запускаешь ноутбук из папки notebooks, то PROJECT_ROOT = Path("..")
# Если запускаешь из корня проекта, то можно поставить PROJECT_ROOT = Path(".")
PROJECT_ROOT = Path("..").resolve()

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_RAW = PROJECT_ROOT / "data" / "raw"

TAXI_PATH = DATA_PROCESSED / "my_clean_2.parquet"

WEATHER_2023_PATH = DATA_RAW / "open-meteo-40.74N74.04W32m.csv"
WEATHER_2024_PATH = DATA_RAW / "open-meteo-40.74N74.04W32m (1).csv"

MODELS_DIR = PROJECT_ROOT / "models"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

MODELS_DIR.mkdir(exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TAXI_PATH exists:", TAXI_PATH.exists(), TAXI_PATH)
print("WEATHER_2023 exists:", WEATHER_2023_PATH.exists(), WEATHER_2023_PATH)
print("WEATHER_2024 exists:", WEATHER_2024_PATH.exists(), WEATHER_2024_PATH)


## 2. Читаем погоду за 2023 и 2024 год

Файлы Open-Meteo имеют особенность: первые строки — это метаданные про координаты, а нормальная таблица начинается не с первой строки.

Поэтому читаем так:

```python
pd.read_csv(path, skiprows=3)
```

То есть пропускаем первые 3 строки.


In [ ]:
def load_open_meteo(path: Path) -> pd.DataFrame:
    """Читает CSV Open-Meteo и приводит названия колонок к простому виду."""
    df = pd.read_csv(path, skiprows=3)

    # Переименовываем длинные названия колонок в короткие.
    rename_map = {
        "time": "date_hour",
        "temperature_2m (°C)": "temperature",
        "relative_humidity_2m (%)": "humidity",
        "precipitation (mm)": "precipitation",
        "rain (mm)": "rain",
        "snowfall (cm)": "snowfall",
        "weather_code (wmo code)": "weather_code",
        "cloud_cover (%)": "cloud_cover",
        "wind_speed_10m (km/h)": "wind_speed",
        "wind_direction_10m (°)": "wind_direction",
        "wind_gusts_10m (km/h)": "wind_gusts",
        "surface_pressure (hPa)": "pressure",
    }
    df = df.rename(columns=rename_map)

    # Переводим время в datetime.
    # В Open-Meteo время уже почасовое.
    df["date_hour"] = pd.to_datetime(df["date_hour"])

    # Оставляем только нужные погодные колонки.
    weather_cols = [
        "date_hour",
        "temperature",
        "humidity",
        "precipitation",
        "rain",
        "snowfall",
        "weather_code",
        "cloud_cover",
        "wind_speed",
        "wind_direction",
        "wind_gusts",
        "pressure",
    ]

    df = df[weather_cols].copy()

    # На всякий случай сортируем.
    df = df.sort_values("date_hour").reset_index(drop=True)
    return df


weather_2023 = load_open_meteo(WEATHER_2023_PATH)
weather_2024 = load_open_meteo(WEATHER_2024_PATH)

weather = pd.concat([weather_2023, weather_2024], ignore_index=True)
weather = weather.drop_duplicates(subset=["date_hour"]).sort_values("date_hour").reset_index(drop=True)

print(weather.shape)
weather.head()


In [ ]:
print("Минимальная дата погоды:", weather["date_hour"].min())
print("Максимальная дата погоды:", weather["date_hour"].max())
print()
print(weather.isna().sum())


## 3. Читаем исходную таблицу поездок

Для первого запуска можно взять не весь файл, а только часть строк.  
Так быстрее проверить, что весь код работает.

Когда всё заработает, поставь:

```python
MAX_ROWS = None
```

Тогда ноутбук будет обучаться на всей таблице.


In [ ]:
MAX_ROWS = 1_000_000
# MAX_ROWS = None  # включи это потом, когда проверишь, что всё запускается

# Нам нужны не все колонки, а только основные:
# - время посадки;
# - район посадки;
# - сумма поездки;
# - расстояние;
# - число пассажиров.
needed_columns = [
    "tpep_pickup_datetime",
    "PULocationID",
    "total_amount",
    "trip_distance",
    "passenger_count",
]

try:
    taxi = pd.read_parquet(TAXI_PATH, columns=needed_columns)
except Exception as e:
    print("Не получилось прочитать только выбранные колонки.")
    print("Читаю весь parquet целиком.")
    print("Ошибка была:", e)
    taxi = pd.read_parquet(TAXI_PATH)

if MAX_ROWS is not None:
    taxi = taxi.head(MAX_ROWS).copy()

print(taxi.shape)
taxi.head()


In [ ]:
print(taxi.dtypes)
print()
print(taxi.isna().sum())


## 4. Готовим дату и чистим явные ошибки

Что делаем:

1. переводим `tpep_pickup_datetime` в формат даты;
2. округляем время до часа;
3. удаляем строки без района, времени и цены;
4. убираем совсем странные значения, например отрицательную выручку.


In [ ]:
taxi = taxi.copy()

taxi["tpep_pickup_datetime"] = pd.to_datetime(taxi["tpep_pickup_datetime"], errors="coerce")

# Округляем вниз до часа.
# Например:
# 2024-01-01 15:37:00 -> 2024-01-01 15:00:00
taxi["date_hour"] = taxi["tpep_pickup_datetime"].dt.floor("h")

# Удаляем строки, где нет важных значений.
taxi = taxi.dropna(subset=["date_hour", "PULocationID", "total_amount"])

# Приводим район к int.
taxi["PULocationID"] = taxi["PULocationID"].astype(int)

# Убираем странные значения.
taxi = taxi[taxi["total_amount"] >= 0]
taxi = taxi[taxi["trip_distance"].fillna(0) >= 0]

print(taxi.shape)
print("Минимальная дата поездок:", taxi["date_hour"].min())
print("Максимальная дата поездок:", taxi["date_hour"].max())
taxi.head()


## 5. Делаем таблицу спроса и выручки по часам

Теперь агрегируем поездки.

Было: одна строка = одна поездка.  
Станет: одна строка = один район в один час.

Целевые переменные:

```text
demand = количество поездок
revenue = сумма total_amount
```


In [ ]:
hourly = (
    taxi
    .groupby(["date_hour", "PULocationID"])
    .agg(
        demand=("PULocationID", "size"),
        revenue=("total_amount", "sum"),
        avg_total_amount=("total_amount", "mean"),
        avg_trip_distance=("trip_distance", "mean"),
        avg_passenger_count=("passenger_count", "mean"),
    )
    .reset_index()
)

hourly = hourly.sort_values(["date_hour", "PULocationID"]).reset_index(drop=True)

print(hourly.shape)
hourly.head()


## 6. Добавляем погоду к поездкам

Погода у нас тоже почасовая, поэтому соединяем по колонке:

```text
date_hour
```

То есть к каждому району в конкретный час добавится погода в этот час.


In [ ]:
hourly = hourly.merge(weather, on="date_hour", how="left")

print(hourly.shape)
hourly.head()


In [ ]:
weather_cols = [
    "temperature",
    "humidity",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "wind_speed",
    "wind_direction",
    "wind_gusts",
    "pressure",
]

print("Сколько пропусков в погоде после merge:")
print(hourly[weather_cols].isna().sum())

# Если какие-то часы не нашли погоду, заполним ближайшими известными значениями.
# Обычно для 2023-2024 пропусков быть не должно, если поездки тоже за эти годы.
hourly = hourly.sort_values("date_hour").reset_index(drop=True)
hourly[weather_cols] = hourly[weather_cols].ffill().bfill()


## 7. Создаём признаки для модели

Модель не понимает дату как человек, поэтому из даты делаем числовые признаки:

- год;
- месяц;
- день месяца;
- час;
- день недели;
- выходной или нет;
- час пик или нет.

Также добавим признаки из прошлого:

- спрос в этом районе 1 час назад;
- выручка 1 час назад;
- спрос 24 часа назад;
- выручка 24 часа назад;
- средний спрос за последние 6 часов;
- средняя выручка за последние 6 часов;
- средний спрос за последние 24 часа;
- средняя выручка за последние 24 часа.

Это важно, потому что спрос обычно зависит от прошлых значений.


In [ ]:
hourly = hourly.sort_values(["PULocationID", "date_hour"]).reset_index(drop=True)

# Календарные признаки.
hourly["year"] = hourly["date_hour"].dt.year
hourly["month"] = hourly["date_hour"].dt.month
hourly["day"] = hourly["date_hour"].dt.day
hourly["hour"] = hourly["date_hour"].dt.hour
hourly["weekday"] = hourly["date_hour"].dt.weekday  # 0 = понедельник, 6 = воскресенье
hourly["is_weekend"] = hourly["weekday"].isin([5, 6]).astype(int)

# Час пик: утром 7-10 и вечером 16-19.
hourly["is_rush_hour"] = hourly["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

# Лаги по каждому району.
# shift(1) значит значение в предыдущей строке этого же района.
hourly["demand_lag_1h"] = hourly.groupby("PULocationID")["demand"].shift(1)
hourly["revenue_lag_1h"] = hourly.groupby("PULocationID")["revenue"].shift(1)

# Если данные строго почасовые для каждого района, shift(24) будет примерно вчера в этот же час.
hourly["demand_lag_24h"] = hourly.groupby("PULocationID")["demand"].shift(24)
hourly["revenue_lag_24h"] = hourly.groupby("PULocationID")["revenue"].shift(24)

# Скользящие средние.
# Важно: сначала shift(1), чтобы модель не видела текущий demand/revenue.
hourly["demand_roll_mean_6h"] = (
    hourly.groupby("PULocationID")["demand"]
    .shift(1)
    .rolling(window=6, min_periods=1)
    .mean()
)

hourly["revenue_roll_mean_6h"] = (
    hourly.groupby("PULocationID")["revenue"]
    .shift(1)
    .rolling(window=6, min_periods=1)
    .mean()
)

hourly["demand_roll_mean_24h"] = (
    hourly.groupby("PULocationID")["demand"]
    .shift(1)
    .rolling(window=24, min_periods=1)
    .mean()
)

hourly["revenue_roll_mean_24h"] = (
    hourly.groupby("PULocationID")["revenue"]
    .shift(1)
    .rolling(window=24, min_periods=1)
    .mean()
)

# В первых строках для района лагов ещё нет, поэтому ставим 0.
lag_cols = [
    "demand_lag_1h",
    "revenue_lag_1h",
    "demand_lag_24h",
    "revenue_lag_24h",
    "demand_roll_mean_6h",
    "revenue_roll_mean_6h",
    "demand_roll_mean_24h",
    "revenue_roll_mean_24h",
]

hourly[lag_cols] = hourly[lag_cols].fillna(0)

hourly.head()


## 8. Делим на train и test

Для временных данных нельзя просто случайно перемешать строки.

Почему?

Потому что модель должна учиться на прошлом и проверяться на будущем.

Поэтому делаем так:

```text
первые 80% часов  -> train
последние 20% часов -> test
```

Это честная проверка: модель не видит будущее во время обучения.


In [ ]:
# Берём все уникальные часы и сортируем их.
unique_hours = np.array(sorted(hourly["date_hour"].unique()))

split_index = int(len(unique_hours) * 0.8)
split_time = unique_hours[split_index]

print("Всего уникальных часов:", len(unique_hours))
print("Граница train/test:", split_time)

train = hourly[hourly["date_hour"] < split_time].copy()
test = hourly[hourly["date_hour"] >= split_time].copy()

print("Train shape:", train.shape)
print("Test shape:", test.shape)

print()
print("Train period:", train["date_hour"].min(), "—", train["date_hour"].max())
print("Test period:", test["date_hour"].min(), "—", test["date_hour"].max())


In [ ]:
# Сохраняем разделённые выборки.
train_path = TABLES_DIR / "train_hourly_with_weather.parquet"
test_path = TABLES_DIR / "test_hourly_with_weather.parquet"

train.to_parquet(train_path, index=False)
test.to_parquet(test_path, index=False)

print("Train saved to:", train_path)
print("Test saved to:", test_path)


## 9. Выбираем признаки и целевые переменные

Будем обучать две отдельные модели:

1. `demand_model` — предсказывает спрос;
2. `revenue_model` — предсказывает выручку.

`PULocationID` — категориальный признак, потому что номер района не является обычным числом.  
Например, район 100 не “в два раза больше”, чем район 50. Это просто код района.

Поэтому для него используем `OneHotEncoder`.


In [ ]:
target_demand = "demand"
target_revenue = "revenue"

categorical_features = ["PULocationID"]

numeric_features = [
    "year",
    "month",
    "day",
    "hour",
    "weekday",
    "is_weekend",
    "is_rush_hour",

    "avg_trip_distance",
    "avg_passenger_count",

    "temperature",
    "humidity",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "wind_speed",
    "wind_direction",
    "wind_gusts",
    "pressure",

    "demand_lag_1h",
    "revenue_lag_1h",
    "demand_lag_24h",
    "revenue_lag_24h",
    "demand_roll_mean_6h",
    "revenue_roll_mean_6h",
    "demand_roll_mean_24h",
    "revenue_roll_mean_24h",
]

features = categorical_features + numeric_features

X_train = train[features].copy()
X_test = test[features].copy()

y_train_demand = train[target_demand].copy()
y_test_demand = test[target_demand].copy()

y_train_revenue = train[target_revenue].copy()
y_test_revenue = test[target_revenue].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)


## 10. Обучаем модели

Для первого варианта используем `HistGradientBoostingRegressor`.

Это нормальная модель для табличных данных:

- умеет ловить нелинейные зависимости;
- обычно работает лучше простой линейной регрессии;
- быстрее, чем огромный Random Forest на больших данных.

Чтобы модель не давала отрицательные прогнозы, обучаем её на `log1p(y)`:

```python
log1p(y) = log(1 + y)
```

А после прогноза возвращаем обратно через:

```python
expm1(pred) = exp(pred) - 1
```


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("location", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features),
    ],
    remainder="drop",
)

def make_model():
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", HistGradientBoostingRegressor(
                max_iter=250,
                learning_rate=0.06,
                max_leaf_nodes=31,
                random_state=42,
            )),
        ]
    )


demand_model = make_model()
revenue_model = make_model()

# Логарифмируем таргеты.
y_train_demand_log = np.log1p(y_train_demand)
y_train_revenue_log = np.log1p(y_train_revenue)

print("Обучаем модель спроса...")
demand_model.fit(X_train, y_train_demand_log)

print("Обучаем модель выручки...")
revenue_model.fit(X_train, y_train_revenue_log)

print("Готово.")


## 11. Проверяем качество на test

Метрики:

- `MAE` — средняя абсолютная ошибка;
- `RMSE` — среднеквадратичная ошибка, сильнее штрафует большие ошибки;
- `R2` — насколько модель лучше среднего прогноза.

Чем меньше `MAE` и `RMSE`, тем лучше.  
Чем ближе `R2` к 1, тем лучше.


In [ ]:
def evaluate_model(model, X, y_true, title: str):
    pred_log = model.predict(X)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)

    mae = mean_absolute_error(y_true, pred)
    rmse = np.sqrt(mean_squared_error(y_true, pred))
    r2 = r2_score(y_true, pred)

    print(title)
    print("MAE :", round(mae, 3))
    print("RMSE:", round(rmse, 3))
    print("R2  :", round(r2, 3))

    return pred


test_pred_demand = evaluate_model(demand_model, X_test, y_test_demand, "Demand model")
print()
test_pred_revenue = evaluate_model(revenue_model, X_test, y_test_revenue, "Revenue model")


In [ ]:
# Сохраняем предсказания на test, чтобы потом можно было посмотреть ошибки.
test_predictions = test[["date_hour", "PULocationID", "demand", "revenue"]].copy()
test_predictions["predicted_demand"] = test_pred_demand
test_predictions["predicted_revenue"] = test_pred_revenue
test_predictions["demand_error"] = test_predictions["predicted_demand"] - test_predictions["demand"]
test_predictions["revenue_error"] = test_predictions["predicted_revenue"] - test_predictions["revenue"]

test_predictions_path = TABLES_DIR / "test_predictions_with_weather.csv"
test_predictions.to_csv(test_predictions_path, index=False)

print("Saved to:", test_predictions_path)
test_predictions.head()


## 12. Сохраняем модели

После сохранения их можно будет использовать без повторного обучения.


In [ ]:
demand_model_path = MODELS_DIR / "demand_model_with_weather.joblib"
revenue_model_path = MODELS_DIR / "revenue_model_with_weather.joblib"

joblib.dump(demand_model, demand_model_path)
joblib.dump(revenue_model, revenue_model_path)

print("Demand model saved to:", demand_model_path)
print("Revenue model saved to:", revenue_model_path)


## 13. Смотрим, какие районы модель считает самыми прибыльными на test

Это не прогноз следующего часа, а проверка на тестовой части:  
какие строки модель оценила как самые прибыльные.


In [ ]:
top_test_revenue = (
    test_predictions
    .sort_values("predicted_revenue", ascending=False)
    .head(20)
)

top_test_revenue_path = TABLES_DIR / "top20_test_predicted_revenue_with_weather.csv"
top_test_revenue.to_csv(top_test_revenue_path, index=False)

print("Saved to:", top_test_revenue_path)
top_test_revenue


## 14. Делаем прогноз на следующий час

Теперь делаем практический прогноз:

```text
какой район в следующий час даст больше спроса и выручки
```

Для этого создаём таблицу, где:

- одна строка = один район;
- время = следующий час после максимальной даты в данных;
- погода = погода на этот час, если она есть;
- лаги = значения из прошлых часов.


In [ ]:
forecast_hour = hourly["date_hour"].max() + pd.Timedelta(hours=1)
locations = sorted(hourly["PULocationID"].unique())

future = pd.DataFrame({
    "date_hour": [forecast_hour] * len(locations),
    "PULocationID": locations,
})

# Календарные признаки для будущего часа.
future["year"] = future["date_hour"].dt.year
future["month"] = future["date_hour"].dt.month
future["day"] = future["date_hour"].dt.day
future["hour"] = future["date_hour"].dt.hour
future["weekday"] = future["date_hour"].dt.weekday
future["is_weekend"] = future["weekday"].isin([5, 6]).astype(int)
future["is_rush_hour"] = future["hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)

future.head()


In [ ]:
# Добавляем погоду на forecast_hour.
future = future.merge(weather, on="date_hour", how="left")

# Если погоды именно на forecast_hour нет, берём последнюю доступную погоду.
if future[weather_cols].isna().any().any():
    last_weather = weather.sort_values("date_hour").iloc[-1][weather_cols]
    for col in weather_cols:
        future[col] = future[col].fillna(last_weather[col])

future[["date_hour", "PULocationID"] + weather_cols].head()


In [ ]:
# Делаем лаги для будущего часа вручную.

last_1h = (
    hourly[hourly["date_hour"] == forecast_hour - pd.Timedelta(hours=1)]
    [["PULocationID", "demand", "revenue"]]
    .rename(columns={
        "demand": "demand_lag_1h",
        "revenue": "revenue_lag_1h",
    })
)

last_24h = (
    hourly[hourly["date_hour"] == forecast_hour - pd.Timedelta(hours=24)]
    [["PULocationID", "demand", "revenue"]]
    .rename(columns={
        "demand": "demand_lag_24h",
        "revenue": "revenue_lag_24h",
    })
)

recent_6h = (
    hourly[
        (hourly["date_hour"] >= forecast_hour - pd.Timedelta(hours=6)) &
        (hourly["date_hour"] < forecast_hour)
    ]
    .groupby("PULocationID")
    .agg(
        demand_roll_mean_6h=("demand", "mean"),
        revenue_roll_mean_6h=("revenue", "mean"),
    )
    .reset_index()
)

recent_24h = (
    hourly[
        (hourly["date_hour"] >= forecast_hour - pd.Timedelta(hours=24)) &
        (hourly["date_hour"] < forecast_hour)
    ]
    .groupby("PULocationID")
    .agg(
        demand_roll_mean_24h=("demand", "mean"),
        revenue_roll_mean_24h=("revenue", "mean"),
    )
    .reset_index()
)

# Средние значения по расстоянию и пассажирам берём из истории района.
location_means = (
    hourly
    .groupby("PULocationID")
    .agg(
        avg_trip_distance=("avg_trip_distance", "mean"),
        avg_passenger_count=("avg_passenger_count", "mean"),
    )
    .reset_index()
)

future = future.merge(last_1h, on="PULocationID", how="left")
future = future.merge(last_24h, on="PULocationID", how="left")
future = future.merge(recent_6h, on="PULocationID", how="left")
future = future.merge(recent_24h, on="PULocationID", how="left")
future = future.merge(location_means, on="PULocationID", how="left")

# Заполняем пропуски нулями.
future[lag_cols] = future[lag_cols].fillna(0)
future[["avg_trip_distance", "avg_passenger_count"]] = future[["avg_trip_distance", "avg_passenger_count"]].fillna(0)

future.head()


In [ ]:
X_future = future[features].copy()

future_pred_demand = np.expm1(demand_model.predict(X_future))
future_pred_revenue = np.expm1(revenue_model.predict(X_future))

future["predicted_demand"] = np.clip(future_pred_demand, 0, None)
future["predicted_revenue"] = np.clip(future_pred_revenue, 0, None)

future_predictions = future[
    ["date_hour", "PULocationID", "predicted_demand", "predicted_revenue"]
].copy()

future_predictions = future_predictions.sort_values("predicted_revenue", ascending=False)

future_predictions_path = TABLES_DIR / "next_hour_predictions_with_weather.csv"
future_predictions.to_csv(future_predictions_path, index=False)

print("Forecast hour:", forecast_hour)
print("Saved to:", future_predictions_path)

future_predictions.head(20)


## 15. Итог: где больше спрос и где больше выручка

Сохраняем два отдельных топа:

- топ-20 районов по прогнозному спросу;
- топ-20 районов по прогнозной выручке.


In [ ]:
top20_by_revenue = future_predictions.sort_values("predicted_revenue", ascending=False).head(20)
top20_by_demand = future_predictions.sort_values("predicted_demand", ascending=False).head(20)

top20_revenue_path = TABLES_DIR / "top20_next_hour_by_revenue_with_weather.csv"
top20_demand_path = TABLES_DIR / "top20_next_hour_by_demand_with_weather.csv"

top20_by_revenue.to_csv(top20_revenue_path, index=False)
top20_by_demand.to_csv(top20_demand_path, index=False)

print("Top by revenue saved to:", top20_revenue_path)
print("Top by demand saved to:", top20_demand_path)


In [ ]:
top20_by_revenue


In [ ]:
top20_by_demand


## 16. График: факт против прогноза

Для быстрой проверки построим график на первых 200 строках теста.

Если точки примерно рядом с диагональной линией, модель работает нормально.  
Если всё сильно далеко — модель ошибается.


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_demand.head(200), test_pred_demand[:200], alpha=0.6)
plt.xlabel("Фактический спрос")
plt.ylabel("Прогнозный спрос")
plt.title("Спрос: факт против прогноза")
plt.grid(True)

fig_path = FIGURES_DIR / "demand_fact_vs_pred.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved to:", fig_path)


In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test_revenue.head(200), test_pred_revenue[:200], alpha=0.6)
plt.xlabel("Фактическая выручка")
plt.ylabel("Прогнозная выручка")
plt.title("Выручка: факт против прогноза")
plt.grid(True)

fig_path = FIGURES_DIR / "revenue_fact_vs_pred.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved to:", fig_path)


# Что важно запомнить

1. Исходная таблица поездок сама по себе не является готовой таблицей для прогноза спроса.
2. Сначала нужно агрегировать поездки:
   ```text
   район + час -> спрос + выручка
   ```
3. Погоду добавляем по часу:
   ```text
   date_hour
   ```
4. Train/test делим по времени, а не случайно.
5. Модель обучается на прошлом и проверяется на будущем.
6. Итоговый результат — таблица районов с прогнозом спроса и выручки.
